In [ ]:
import nest_asyncio
import asyncio
import pandas as pd
import re
from playwright.async_api import async_playwright

# Apply nest_asyncio to allow Playwright to run in the Colab/Jupyter environment
nest_asyncio.apply()

# ==========================================
# 2. THE SCRAPER ENGINE
# ==========================================
async def scrape_flipkart_market_data(brand_links):
    all_data = []

    async with async_playwright() as p:
        # Launch browser in headless mode (required for Colab)
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
        )

        for brand, base_url in brand_links.items():
            print(f"\nStarting search for {brand} laptops...")
            page = await context.new_page()

            # Loop through pages 1 to 15 (covers ~360 records per brand)
            for page_num in range(1, 26):
                current_url = f"{base_url}&page={page_num}"
                print(f"   - Processing Page {page_num}...")

                try:
                    # Navigate with a generous timeout for Indian e-commerce loads
                    try:
                      # 1. Wait for the initial load
                      await page.goto(current_url, wait_until="load", timeout=60000)
                      # 2. Wait for the network to stop being busy (critical for Flipkart)
                      await page.wait_for_load_state("networkidle")
                      # 3. An extra buffer for slower Colab connections
                      await page.wait_for_timeout(3000)
                    except Exception as e:
                      print(f"Initial load failed, retrying... {e}")
                      await page.reload(wait_until="networkidle")

                    # Close potential login popups that block the view
                    try: await page.keyboard.press("Escape")
                    except: pass

                    # Locate product rows based on the 'ZFwe0M' class you found
                    rows = await page.query_selector_all('div.ZFwe0M.row')

                    if not rows:
                        print(f"No more results found for {brand}. Finalizing list.")
                        break

                    for row in rows:
                        try:
                            # TITLE & RATING
                            name = await (await row.query_selector('div.RG5Slk')).inner_text()
                            rating_el = await row.query_selector('div.MKiFS6')
                            rating = await rating_el.inner_text() if rating_el else "0"

                            # PRICING (Selling, MRP, Discount)
                            sell_el = await row.query_selector('div.hZ3P6w')
                            mrp_el = await row.query_selector('div.kRYCnD')
                            disc_el = await row.query_selector('div.HQe8jr')

                            sell_text = await sell_el.inner_text() if sell_el else "0"
                            mrp_text = await mrp_el.inner_text() if mrp_el else sell_text
                            disc_text = await disc_el.inner_text() if disc_el else "0% off"

                            # AVAILABILITY & IMAGE URL
                            # 1. Grab the element using the class you identified
                            status_el = await row.query_selector('span.fRrrYo')

                            # 2. Get the text if the element exists, otherwise keep it empty
                            status_text = await status_el.inner_text() if status_el else ""

                            # 3. Check if EITHER phrase is present
                            if "Currently unavailable" in status_text or "Not deliverable" in status_text:
                                availability = "Unavailable"
                            else:
                                availability = "Available"
                            ##unavailable_el = await row.query_selector('span.fRrrYo')
                            ##availability = "Unavailable" if unavailable_el and "Currently unavailable" in (await unavailable_el.inner_text()) else "Available"

                            img_el = await row.query_selector('img.UCc1lI')
                            img_url = await img_el.get_attribute('src') if img_el else "No Image Link"

                            # SPECS EXTRACTION (Joining bullets with '|')
                            spec_items = await row.query_selector_all('li.DTBslk')
                            specs = " | ".join([await s.inner_text() for s in spec_items])

                            # DATA CLEANING FOR ANALYSIS
                            # Remove non-numeric characters like ₹ and commas
                            clean_sell = int(re.sub(r'[^\d]', '', sell_text)) if sell_text != "0" else 0
                            clean_mrp = int(re.sub(r'[^\d]', '', mrp_text)) if mrp_text != "0" else clean_sell
                            # Extract just the number from "25% off"
                            clean_disc = int(re.search(r'\d+', disc_text).group()) if re.search(r'\d+', disc_text) else 0

                            all_data.append({
                                "Brand": brand,
                                "Model": name,
                                "Selling_Price": clean_sell,
                                "Original_MRP": clean_mrp,
                                "Discount_Percentage": clean_disc,
                                "Customer_Rating": float(rating.strip()),
                                "Availability": availability,
                                "Specifications": specs,
                                "Image_Source": img_url
                            })
                        except Exception:
                            continue # Skip row if it has missing data or structural issues

                except Exception as e:
                    print(f"Error on page {page_num}: {e}")
                    break

            await page.close()

        await browser.close()

    return pd.DataFrame(all_data)

# ==========================================
# 3. CONFIGURE LINKS & EXECUTE
# ==========================================
# These are your filtered links including "Out of Stock"
brand_urls = {
    "Lenovo": "https://www.flipkart.com/search?q=laptop&p%5B%5D=facets.brand%255B%255D%3DLenovo&p%5B%5D=facets.availability%255B%255D%3DInclude%2BOut%2Bof%2BStock",
    "Dell": "https://www.flipkart.com/search?q=laptop&p%5B%5D=facets.brand%255B%255D%3DDELL&p%5B%5D=facets.availability%255B%255D%3DInclude%2BOut%2Bof%2BStock",
    "ASUS": "https://www.flipkart.com/search?q=laptop&p%5B%5D=facets.brand%255B%255D%3DASUS&p%5B%5D=facets.availability%255B%255D%3DInclude%2BOut%2Bof%2BStock",
    "HP": "https://www.flipkart.com/search?q=laptop&p%5B%5D=facets.brand%255B%255D%3DHP&p%5B%5D=facets.availability%255B%255D%3DInclude%2BOut%2Bof%2BStock"
}

# Run the async scraper
df = asyncio.run(scrape_flipkart_market_data(brand_urls))

# ==========================================
# 4. FINAL OUTPUT & ANALYSIS
# ==========================================
if not df.empty:
    # Save the master dataset
    df.to_csv("flipkart_laptop_analysis_2026_1.csv", index=False)
    print("\nSUCCESS: Data saved to 'flipkart_laptop_analysis_2026.csv'")

    # Perform Quick Data Aggregations
    brand_summary = df.groupby('Brand').agg({
        'Selling_Price': 'mean',
        'Discount_Percentage': 'mean',
        'Customer_Rating': 'mean'
    }).round(2).rename(columns={'Selling_Price': 'Avg_Selling_Price', 'Discount_Percentage': 'Avg_Discount_%'})

    print("\n--- BRAND PERFORMANCE SUMMARY ---")
    print(brand_summary)

    print("\n--- OUT-OF-STOCK SNAPSHOT ---")
    print(df[df['Availability'] == 'Unavailable'][['Brand', 'Model', 'Selling_Price']].head(10))
else:
    print("\nFAILED: No data was collected. Please check your internet connection or selectors.")


🚀 Starting search for Lenovo laptops...
   - Processing Page 1...
   - Processing Page 2...
   - Processing Page 3...
   - Processing Page 4...
   - Processing Page 5...
   - Processing Page 6...
   - Processing Page 7...
   - Processing Page 8...
   - Processing Page 9...
   - Processing Page 10...
   - Processing Page 11...
   - Processing Page 12...
   - Processing Page 13...
   - Processing Page 14...
   - Processing Page 15...
   - Processing Page 16...
   - Processing Page 17...
   - Processing Page 18...
   - Processing Page 19...
   - Processing Page 20...
   - Processing Page 21...
   - Processing Page 22...
   - Processing Page 23...
   - Processing Page 24...
   - Processing Page 25...

🚀 Starting search for Dell laptops...
   - Processing Page 1...
   - Processing Page 2...
   - Processing Page 3...
   - Processing Page 4...
   - Processing Page 5...
   - Processing Page 6...
   - Processing Page 7...
   - Processing Page 8...
   - Processing Page 9...
   - Processing Page

In [ ]:
import re

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

uploaded = files.upload()

Saving flipkart_laptop_analysis_2026_2.csv to flipkart_laptop_analysis_2026_2 (1).csv


In [ ]:
laptop_df= pd.read_csv('flipkart_laptop_analysis_2026_2.csv')

In [ ]:
# Checking first 5 rows for understanding the structure
laptop_df.head()

,Brand,Model,Selling_Price,Original_MRP,Discount_Percentage,Customer_Rating,Availability,Specifications,Image_Source
0,Lenovo,Lenovo IdeaPad Slim 3 AMD Ryzen 7 Octa Core 88...,83990,94890,11,4.4,Available,AMD Ryzen 7 Octa Core Processor | 24 GB DDR5 R...,No Image Link
1,Lenovo,Lenovo IdeaPad Slim 3 Backlit Intel Core i5 12...,54990,72690,24,4.2,Available,Intel Core i5 Processor (12th Gen) | 16 GB LPD...,No Image Link
2,Lenovo,Lenovo 100e Chromebook Gen 4 MediaTek Kompanio...,15990,16298,1,4.0,Available,MediaTek Kompanio 520 Processor | 4 GB LPDDR4X...,No Image Link
3,Lenovo,Lenovo �IdeaPad Slim 5 WUXGA OLED Full Metal B...,69990,85390,18,4.4,Available,Intel Core i5 Processor (13th Gen) | 16 GB DDR...,No Image Link
4,Lenovo,Lenovo IdeaPad Slim 3 15ABR8 AMD Ryzen 5 Hexa ...,53142,62990,15,4.5,Available,AMD Ryzen 5 Hexa Core Processor | 16 GB DDR4 R...,No Image Link


In [ ]:
# Checking for NaN values per column
laptop_df.isna().sum()

,0
Brand,0
Model,0
Selling_Price,0
Original_MRP,0
Discount_Percentage,0
Customer_Rating,0
Availability,0
Specifications,0
Image_Source,0


In [ ]:
# Checking for the datatypes for each column value
laptop_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2352 entries, 0 to 2351
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Brand                2352 non-null   object 
 1   Model                2352 non-null   object 
 2   Selling_Price        2352 non-null   int64  
 3   Original_MRP         2352 non-null   int64  
 4   Discount_Percentage  2352 non-null   int64  
 5   Customer_Rating      2352 non-null   float64
 6   Availability         2352 non-null   object 
 7   Specifications       2352 non-null   object 
 8   Image_Source         2352 non-null   object 
dtypes: float64(1), int64(3), object(5)
memory usage: 165.5+ KB


In [ ]:
# further opearions will be done on lpdf (a copy of laptop_df dataframe) as we don't want to change the original dataset for backup
lpdf=laptop_df.copy()

In [ ]:
lpdf["Model"]

,Model
0,Lenovo IdeaPad Slim 3 AMD Ryzen 7 Octa Core 88...
1,Lenovo IdeaPad Slim 3 Backlit Intel Core i5 12...
2,Lenovo 100e Chromebook Gen 4 MediaTek Kompanio...
3,Lenovo �IdeaPad Slim 5 WUXGA OLED Full Metal B...
4,Lenovo IdeaPad Slim 3 15ABR8 AMD Ryzen 5 Hexa ...
...,...
2347,HP Laptop Intel Core i5 12th Gen 1235U - (8 GB...
2348,HP Intel Core i5 14th Gen 14450HX - (16 GB/512...
2349,HP AMD Ryzen 5 Hexa Core 5500U - (8 GB/512 GB ...
2350,HP 14s AMD Ryzen 5 Quad Core 3450U - (8 GB/512...


In [ ]:
lpdf["Specifications"]

,Specifications
0,AMD Ryzen 7 Octa Core Processor | 24 GB DDR5 R...
1,Intel Core i5 Processor (12th Gen) | 16 GB LPD...
2,MediaTek Kompanio 520 Processor | 4 GB LPDDR4X...
3,Intel Core i5 Processor (13th Gen) | 16 GB DDR...
4,AMD Ryzen 5 Hexa Core Processor | 16 GB DDR4 R...
...,...
2347,Intel Core i5 Processor (12th Gen) | 8 GB DDR4...
2348,Intel Core i5 Processor (14th Gen) | 16 GB LPD...
2349,AMD Ryzen 5 Hexa Core Processor | 8 GB DDR4 RA...
2350,AMD Ryzen 5 Quad Core Processor | 8 GB DDR4 RA...


In [ ]:
# EDA
# 1. Fetch the Model name from the original Model column as it contains device specifications also
# 2. Splitting Specifications into various sub columns (Procesor, RAM, RAM_type, OS, Disk_size and Display_size)

In [ ]:
# EDA Fetch the Model name from the original Model column as it contains device specifications also
# 1. Define the regex 'Border'
# This pattern looks for common technical keywords.
# \b ensures it matches the whole word (so 'i3' won't match 'hi3').
pattern = r'\b(Intel|AMD|Ryzen|Core|i\d|M1|M2|M3|RAM|SSD|DDR|GB|Apple|Octa|Quad|MediaTek|WUXGA)\b'

def clean_laptop_name(text):
    # If the text is empty, just return it
    if pd.isna(text): return text

    # Search for the first occurrence of a spec keyword
    match = re.search(pattern, text, flags=re.IGNORECASE)

    if match:
        # Get everything before the keyword started
        clean_name = text[:match.start()].strip()

        # Snip off trailing symbols like - or ,
        clean_name = re.sub(r'[-–, ]+$', '', clean_name)
        return clean_name

    # If no keywords found, it's already a clean name
    return text

# 2. Apply clean_laptop_name function
lpdf['Model'] = lpdf['Model'].apply(clean_laptop_name)

# 3. Check your results
print(lpdf[['Model']].head(10))

                           Model
0          Lenovo IdeaPad Slim 3
1  Lenovo IdeaPad Slim 3 Backlit
2   Lenovo 100e Chromebook Gen 4
3         Lenovo �IdeaPad Slim 5
4   Lenovo IdeaPad Slim 3 15ABR8
5          Lenovo IdeaPad Slim 3
6              Lenovo Chromebook
7          Lenovo IdeaPad Slim 3
8           Lenovo IdeaPad Pro 5
9              Lenovo Chromebook


In [ ]:
print(lpdf[['Model']].tail(10))

           Model
2342          HP
2343      HP 15s
2344  HP Probook
2345          HP
2346  HP OMEN 16
2347   HP Laptop
2348          HP
2349          HP
2350      HP 14s
2351      HP 14s


In [ ]:
lpdf.head()

,Brand,Model,Selling_Price,Original_MRP,Discount_Percentage,Customer_Rating,Availability,Specifications,Image_Source
0,Lenovo,Lenovo IdeaPad Slim 3,83990,94890,11,4.4,Available,AMD Ryzen 7 Octa Core Processor | 24 GB DDR5 R...,No Image Link
1,Lenovo,Lenovo IdeaPad Slim 3 Backlit,54990,72690,24,4.2,Available,Intel Core i5 Processor (12th Gen) | 16 GB LPD...,No Image Link
2,Lenovo,Lenovo 100e Chromebook Gen 4,15990,16298,1,4.0,Available,MediaTek Kompanio 520 Processor | 4 GB LPDDR4X...,No Image Link
3,Lenovo,Lenovo �IdeaPad Slim 5,69990,85390,18,4.4,Available,Intel Core i5 Processor (13th Gen) | 16 GB DDR...,No Image Link
4,Lenovo,Lenovo IdeaPad Slim 3 15ABR8,53142,62990,15,4.5,Available,AMD Ryzen 5 Hexa Core Processor | 16 GB DDR4 R...,No Image Link


In [ ]:
# EDA Splitting Specifications into various sub columns (Procesor, RAM, RAM_type, OS, Disk_size and Display_size)

# 1. Clean the 'Specs' column text (remove extra spaces)
# 2. Split by the '|' symbol, limiting to 4 splits to ensure 5 resulting columns
# 3. 'expand=True' turns the result into new columns

specs_df = lpdf['Specifications'].str.split('|', n=4, expand=True)

# 4. Rename the columns so they make sense
specs_df.columns = ['Processor', 'RAM', 'OS', 'Disk_Size', 'Display_Size']

# 5. Clean up whitespace in every new column
for col in specs_df.columns:
    specs_df[col] = specs_df[col].str.strip()

# 6. Join these new columns back to your main lpdf
lpdf = pd.concat([lpdf, specs_df], axis=1)

In [ ]:
lpdf.head()

,Brand,Model,Selling_Price,Original_MRP,Discount_Percentage,Customer_Rating,Availability,Specifications,Image_Source,Processor,RAM,OS,Disk_Size,Display_Size
0,Lenovo,Lenovo IdeaPad Slim 3,83990,94890,11,4.4,Available,AMD Ryzen 7 Octa Core Processor | 24 GB DDR5 R...,No Image Link,AMD Ryzen 7 Octa Core Processor,24 GB DDR5 RAM,Windows 11 Home Operating System,1 TB SSD,38.86 cm (15.3 Inch) Display | Office Home 202...
1,Lenovo,Lenovo IdeaPad Slim 3 Backlit,54990,72690,24,4.2,Available,Intel Core i5 Processor (12th Gen) | 16 GB LPD...,No Image Link,Intel Core i5 Processor (12th Gen),16 GB LPDDR5 RAM,Windows 11 Operating System,512 GB SSD,39.62 cm (15.6 Inch) Display | 1 Yr. Onsite + ...
2,Lenovo,Lenovo 100e Chromebook Gen 4,15990,16298,1,4.0,Available,MediaTek Kompanio 520 Processor | 4 GB LPDDR4X...,No Image Link,MediaTek Kompanio 520 Processor,4 GB LPDDR4X RAM,Chrome Operating System,29.46 cm (11.6 Inch) Display,1 Year Carry-in Warranty
3,Lenovo,Lenovo �IdeaPad Slim 5,69990,85390,18,4.4,Available,Intel Core i5 Processor (13th Gen) | 16 GB DDR...,No Image Link,Intel Core i5 Processor (13th Gen),16 GB DDR5 RAM,Windows 11 Home Operating System,512 GB SSD,35.56 cm (14 Inch) Display | Office Home 2024 ...
4,Lenovo,Lenovo IdeaPad Slim 3 15ABR8,53142,62990,15,4.5,Available,AMD Ryzen 5 Hexa Core Processor | 16 GB DDR4 R...,No Image Link,AMD Ryzen 5 Hexa Core Processor,16 GB DDR4 RAM,Windows 11 Home Operating System,512 GB SSD,39.62 cm (15.6 Inch) Display | Office Home 202...


In [ ]:
ram_type_pattern = r'DDR\d|LPDDR\d'
def get_ram_type(text):
    if pd.isna(text):
        return "Unknown"

    # search looks everywhere in the string, not just the start
    match = re.search(ram_type_pattern, text, flags=re.IGNORECASE)

    if match:
        # .group() returns exactly what it found (e.g., "DDR5")
        return match.group().upper()

    return "Other"

# Apply it to your split RAM column
lpdf['RAM_Type'] = lpdf['RAM'].apply(get_ram_type)

In [ ]:
lpdf.head()

,Brand,Model,Selling_Price,Original_MRP,Discount_Percentage,Customer_Rating,Availability,Specifications,Image_Source,Processor,RAM,OS,Disk_Size,Display_Size,RAM_Type
0,Lenovo,Lenovo IdeaPad Slim 3,83990,94890,11,4.4,Available,AMD Ryzen 7 Octa Core Processor | 24 GB DDR5 R...,No Image Link,AMD Ryzen 7 Octa Core Processor,24 GB DDR5 RAM,Windows 11 Home Operating System,1 TB SSD,38.86 cm (15.3 Inch) Display | Office Home 202...,DDR5
1,Lenovo,Lenovo IdeaPad Slim 3 Backlit,54990,72690,24,4.2,Available,Intel Core i5 Processor (12th Gen) | 16 GB LPD...,No Image Link,Intel Core i5 Processor (12th Gen),16 GB LPDDR5 RAM,Windows 11 Operating System,512 GB SSD,39.62 cm (15.6 Inch) Display | 1 Yr. Onsite + ...,LPDDR5
2,Lenovo,Lenovo 100e Chromebook Gen 4,15990,16298,1,4.0,Available,MediaTek Kompanio 520 Processor | 4 GB LPDDR4X...,No Image Link,MediaTek Kompanio 520 Processor,4 GB LPDDR4X RAM,Chrome Operating System,29.46 cm (11.6 Inch) Display,1 Year Carry-in Warranty,LPDDR4
3,Lenovo,Lenovo �IdeaPad Slim 5,69990,85390,18,4.4,Available,Intel Core i5 Processor (13th Gen) | 16 GB DDR...,No Image Link,Intel Core i5 Processor (13th Gen),16 GB DDR5 RAM,Windows 11 Home Operating System,512 GB SSD,35.56 cm (14 Inch) Display | Office Home 2024 ...,DDR5
4,Lenovo,Lenovo IdeaPad Slim 3 15ABR8,53142,62990,15,4.5,Available,AMD Ryzen 5 Hexa Core Processor | 16 GB DDR4 R...,No Image Link,AMD Ryzen 5 Hexa Core Processor,16 GB DDR4 RAM,Windows 11 Home Operating System,512 GB SSD,39.62 cm (15.6 Inch) Display | Office Home 202...,DDR4


In [ ]:
# This removes 'DDR4', 'DDR5', etc., and also the word 'RAM' if it's there
# The [45]? means it looks for a 4 or a 5 optionally
lpdf['RAM'] = lpdf['RAM'].str.replace(r'DDR[45]?|RAM|LP|LPX|X', '', regex=True, flags=re.IGNORECASE)

# Clean up any leftover double spaces or leading/trailing spaces
lpdf['RAM'] = lpdf['RAM'].str.strip()

print(lpdf['RAM'].head())

0    24 GB
1    16 GB
2     4 GB
3    16 GB
4    16 GB
Name: RAM, dtype: object


In [ ]:
lpdf.head()

,Brand,Model,Selling_Price,Original_MRP,Discount_Percentage,Customer_Rating,Availability,Specifications,Image_Source,Processor,RAM,OS,Disk_Size,Display_Size,RAM_Type
0,Lenovo,Lenovo IdeaPad Slim 3,83990,94890,11,4.4,Available,AMD Ryzen 7 Octa Core Processor | 24 GB DDR5 R...,No Image Link,AMD Ryzen 7 Octa Core Processor,24 GB,Windows 11 Home Operating System,1 TB SSD,38.86 cm (15.3 Inch) Display | Office Home 202...,DDR5
1,Lenovo,Lenovo IdeaPad Slim 3 Backlit,54990,72690,24,4.2,Available,Intel Core i5 Processor (12th Gen) | 16 GB LPD...,No Image Link,Intel Core i5 Processor (12th Gen),16 GB,Windows 11 Operating System,512 GB SSD,39.62 cm (15.6 Inch) Display | 1 Yr. Onsite + ...,LPDDR5
2,Lenovo,Lenovo 100e Chromebook Gen 4,15990,16298,1,4.0,Available,MediaTek Kompanio 520 Processor | 4 GB LPDDR4X...,No Image Link,MediaTek Kompanio 520 Processor,4 GB,Chrome Operating System,29.46 cm (11.6 Inch) Display,1 Year Carry-in Warranty,LPDDR4
3,Lenovo,Lenovo �IdeaPad Slim 5,69990,85390,18,4.4,Available,Intel Core i5 Processor (13th Gen) | 16 GB DDR...,No Image Link,Intel Core i5 Processor (13th Gen),16 GB,Windows 11 Home Operating System,512 GB SSD,35.56 cm (14 Inch) Display | Office Home 2024 ...,DDR5
4,Lenovo,Lenovo IdeaPad Slim 3 15ABR8,53142,62990,15,4.5,Available,AMD Ryzen 5 Hexa Core Processor | 16 GB DDR4 R...,No Image Link,AMD Ryzen 5 Hexa Core Processor,16 GB,Windows 11 Home Operating System,512 GB SSD,39.62 cm (15.6 Inch) Display | Office Home 202...,DDR4


In [ ]:
# This pattern looks for:
# 1. A number (potentially with a decimal like 15.6) -> (\d+\.?\d*)
# 2. Followed by a space (optional) -> \s*
# 3. Followed by "Inch" or "inch" -> [Ii]nch
display_pattern = r'(\d+\.?\d*)\s*[Ii]nch'

def clean_display(text):
    if pd.isna(text):
        return np.nan

    # Search for the pattern anywhere in the messy string
    match = re.search(display_pattern, text)

    if match:
        # Reconstruct it as "Number Inch" for a clean look
        return f"{match.group(1)} Inch"

    return np.nan

# Apply it to your lpdf
lpdf['Display_Size'] = lpdf['Display_Size'].apply(clean_display)

print(lpdf[['Display_Size']].head())

  Display_Size
0    15.3 Inch
1    15.6 Inch
2          NaN
3      14 Inch
4    15.6 Inch


In [ ]:
lpdf.head()

,Brand,Model,Selling_Price,Original_MRP,Discount_Percentage,Customer_Rating,Availability,Specifications,Image_Source,Processor,RAM,OS,Disk_Size,Display_Size,RAM_Type
0,Lenovo,Lenovo IdeaPad Slim 3,83990,94890,11,4.4,Available,AMD Ryzen 7 Octa Core Processor | 24 GB DDR5 R...,No Image Link,AMD Ryzen 7 Octa Core Processor,24 GB,Windows 11 Home Operating System,1 TB SSD,15.3 Inch,DDR5
1,Lenovo,Lenovo IdeaPad Slim 3 Backlit,54990,72690,24,4.2,Available,Intel Core i5 Processor (12th Gen) | 16 GB LPD...,No Image Link,Intel Core i5 Processor (12th Gen),16 GB,Windows 11 Operating System,512 GB SSD,15.6 Inch,LPDDR5
2,Lenovo,Lenovo 100e Chromebook Gen 4,15990,16298,1,4.0,Available,MediaTek Kompanio 520 Processor | 4 GB LPDDR4X...,No Image Link,MediaTek Kompanio 520 Processor,4 GB,Chrome Operating System,29.46 cm (11.6 Inch) Display,NaN,LPDDR4
3,Lenovo,Lenovo �IdeaPad Slim 5,69990,85390,18,4.4,Available,Intel Core i5 Processor (13th Gen) | 16 GB DDR...,No Image Link,Intel Core i5 Processor (13th Gen),16 GB,Windows 11 Home Operating System,512 GB SSD,14 Inch,DDR5
4,Lenovo,Lenovo IdeaPad Slim 3 15ABR8,53142,62990,15,4.5,Available,AMD Ryzen 5 Hexa Core Processor | 16 GB DDR4 R...,No Image Link,AMD Ryzen 5 Hexa Core Processor,16 GB,Windows 11 Home Operating System,512 GB SSD,15.6 Inch,DDR4


In [ ]:
# Pattern Breakdown:
# (\d+)      -> Group 1: Match one or more digits
# \s* -> Match zero or more spaces
# (GB|TB)    -> Group 2: Match exactly 'GB' OR 'TB'
storage_pattern = r'(\d+)\s*(GB|TB)'

def clean_storage(text):
    if pd.isna(text):
        return np.nan

    # search for the pattern anywhere in the string
    match = re.search(storage_pattern, text, flags=re.IGNORECASE)

    if match:
        # Re-construct to a clean, standard format: "Value UNIT"
        # .upper() ensures 'tb' becomes 'TB'
        return f"{match.group(1)} {match.group(2).upper()}"

    # If it's anything else, return NaN
    return np.nan

# Apply to your storage column
lpdf['Disk_Size'] = lpdf['Disk_Size'].apply(clean_storage)

print(lpdf[['Disk_Size']].head())

  Disk_Size
0      1 TB
1    512 GB
2       NaN
3    512 GB
4    512 GB


In [ ]:
lpdf.head()

,Brand,Model,Selling_Price,Original_MRP,Discount_Percentage,Customer_Rating,Availability,Specifications,Image_Source,Processor,RAM,OS,Disk_Size,Display_Size,RAM_Type
0,Lenovo,Lenovo IdeaPad Slim 3,83990,94890,11,4.4,Available,AMD Ryzen 7 Octa Core Processor | 24 GB DDR5 R...,No Image Link,AMD Ryzen 7 Octa Core Processor,24 GB,Windows 11 Home Operating System,1 TB,15.3 Inch,DDR5
1,Lenovo,Lenovo IdeaPad Slim 3 Backlit,54990,72690,24,4.2,Available,Intel Core i5 Processor (12th Gen) | 16 GB LPD...,No Image Link,Intel Core i5 Processor (12th Gen),16 GB,Windows 11 Operating System,512 GB,15.6 Inch,LPDDR5
2,Lenovo,Lenovo 100e Chromebook Gen 4,15990,16298,1,4.0,Available,MediaTek Kompanio 520 Processor | 4 GB LPDDR4X...,No Image Link,MediaTek Kompanio 520 Processor,4 GB,Chrome Operating System,NaN,NaN,LPDDR4
3,Lenovo,Lenovo �IdeaPad Slim 5,69990,85390,18,4.4,Available,Intel Core i5 Processor (13th Gen) | 16 GB DDR...,No Image Link,Intel Core i5 Processor (13th Gen),16 GB,Windows 11 Home Operating System,512 GB,14 Inch,DDR5
4,Lenovo,Lenovo IdeaPad Slim 3 15ABR8,53142,62990,15,4.5,Available,AMD Ryzen 5 Hexa Core Processor | 16 GB DDR4 R...,No Image Link,AMD Ryzen 5 Hexa Core Processor,16 GB,Windows 11 Home Operating System,512 GB,15.6 Inch,DDR4


In [ ]:
#print(f"Unique RAM values: {lpdf['RAM'].unique()}")
#print(f"Unique RAM Type values: {lpdf['RAM_Type'].unique()}")
print(f"Unique Display Size values: {lpdf['Display_Size'].unique()}")

Unique Display Size values: ['15.3 Inch' '15.6 Inch' nan '14 Inch' '13.3 Inch' '11.6 Inch' '16 Inch'
 '14.5 Inch' '15.1 Inch' '15 Inch' '13.9 Inch' '14.2 Inch' '13 Inch'
 '14.1 Inch' '14.9 Inch' '18 Inch' '14.96 Inch' '13.7 Inch' '13.78 Inch'
 '13.4 Inch' '12.9921 Inch' '17.3 Inch' '7 Inch' '15.5 Inch' '16.1 Inch'
 '13.5 Inch' '39.62156 Inch' '33.78 Inch' '16.2 Inch']


In [ ]:
lpdf['OS'].unique()

array(['Windows 11 Home Operating System', 'Windows 11 Operating System',
       'Chrome Operating System', '64 bit Windows 11 Operating System',
       '64 bit Chrome Operating System',
       '64 bit Windows 11 Home Operating System',
       'Windows 10 Operating System',
       '64 bit Windows 10 Operating System', 'DOS Operating System',
       '64 bit DOS Operating System', '16 GB LPDDR4X RAM',
       '4 GB DDR4 RAM', '4 GB DDR3 RAM', 'Linux/Ubuntu Operating System',
       'Graphics & Keyboard: Integrated & Backlit Keyboard+Fingerprint Reader',
       'Display:15.6" FHD WVA AG Narrow Border & Standard Keyboard',
       '4-zone RGB keyboard with WASD and chassis lighting',
       'Graphics & Keyboard: Integrated & Backlit Keyboard + Fingerprint Reader',
       'Display & Graphics: 15.6" FHD WVA AG 120Hz 250 nits Narrow Border & NVIDIA GEFORCE RTX 3050 (4GB GDDR6)',
       'Software: Win 11 + Office H&S 2021',
       'Display: 15.6" HD AG Narrow Border',
       'Graphics & Keyboard

In [ ]:
lpdf.loc[lpdf['OS'] == '16 GB LPDDR4X RAM']

,Brand,Model,Selling_Price,Original_MRP,Discount_Percentage,Customer_Rating,Availability,Specifications,Image_Source,Processor,RAM,OS,Disk_Size,Display_Size,RAM_Type
470,Lenovo,lenovo Yoga 9,172990,237890,27,0.0,Available,Intel Evo platform feat 11th Gen Intel Core i7...,No Image Link,Intel Evo platform feat 11th Gen Intel Core i7...,Intel Core i7 Processor (11th Gen),16 GB LPDDR4X RAM,NaN,14 Inch,Other


In [ ]:
lpdf.iloc[470]['Specifications']

'Intel Evo platform feat 11th Gen Intel Core i7 processor | Intel Core i7 Processor (11th Gen) | 16 GB LPDDR4X RAM | 64 bit Windows 10 Operating System | 1 TB SSD | 35.56 cm (14 Inch) Touchscreen Display | Microsoft Office Home and Student 2019 | 1 Year Onsite Warranty'

In [ ]:
lpdf['OS'].unique()

array(['Windows 11 Home Operating System', 'Windows 11 Operating System',
       'Chrome Operating System', '64 bit Windows 11 Operating System',
       '64 bit Chrome Operating System',
       '64 bit Windows 11 Home Operating System',
       'Windows 10 Operating System',
       '64 bit Windows 10 Operating System', 'DOS Operating System',
       '64 bit DOS Operating System', '16 GB LPDDR4X RAM',
       '4 GB DDR4 RAM', '4 GB DDR3 RAM', 'Linux/Ubuntu Operating System',
       'Graphics & Keyboard: Integrated & Backlit Keyboard+Fingerprint Reader',
       'Display:15.6" FHD WVA AG Narrow Border & Standard Keyboard',
       '4-zone RGB keyboard with WASD and chassis lighting',
       'Graphics & Keyboard: Integrated & Backlit Keyboard + Fingerprint Reader',
       'Display & Graphics: 15.6" FHD WVA AG 120Hz 250 nits Narrow Border & NVIDIA GEFORCE RTX 3050 (4GB GDDR6)',
       'Software: Win 11 + Office H&S 2021',
       'Display: 15.6" HD AG Narrow Border',
       'Graphics & Keyboard

In [ ]:
lpdf['Processor'].unique()

array(['AMD Ryzen 7 Octa Core Processor',
       'Intel Core i5 Processor (12th Gen)',
       'MediaTek Kompanio 520 Processor',
       'Intel Core i5 Processor (13th Gen)',
       'AMD Ryzen 5 Hexa Core Processor', 'Intel Core Ultra 9 Processor',
       'Intel Pentium Quad Core Processor',
       'Intel Celeron Dual Core Processor', 'Snapdragon X Plus Processor',
       'Intel Core i7 Processor',
       'MediaTek MediaTek Kompanio 838 Processor',
       'Intel Core i7 Processor (13th Gen)',
       'Intel Core i3 Processor (13th Gen)',
       'Intel Core i3 Processor (11th Gen)',
       'AMD Ryzen 3 Quad Core Processor',
       'AMD Ryzen 5 Quad Core Processor',
       'Intel Core i3 Processor (12th Gen)', 'Intel Core 5 Processor',
       'Intel Core i5 Processor (11th Gen)',
       'Intel Core i3 Processor (10th Gen)',
       'AMD Ryzen 3 Quad Core Processor (7th Gen)',
       'AMD Ryzen 3 Hexa Core Processor (3rd Gen)',
       'MediaTek MediaTek Kompanio 528 Processor',
       'Intel